# Support Ticket Classification & Prioritization System

## Complete NLP Pipeline Analysis

This notebook demonstrates the complete support ticket classification system including:

1. **Data Loading & Preprocessing**
2. **Feature Extraction**
3. **Model Training & Evaluation**
4. **Business Insights & Recommendations**
5. **Interactive Predictions**

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Import our custom modules
import sys
sys.path.append('../src')

from preprocess import TicketPreprocessor, load_and_preprocess_data
from vectorize import prepare_data_splits, vectorize_data, analyze_features
from train_model import ModelTrainer
from evaluate import ModelEvaluator
from predict import TicketPredictor

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully!")

## 1. Data Loading & Exploration

In [ ]:
# Load and explore the dataset
print("Loading support ticket data...")
df_processed, preprocessor = load_and_preprocess_data('../data/tickets.csv')

# Display basic information
print(f"\nDataset Shape: {df_processed.shape}")
print(f"\nColumns: {list(df_processed.columns)}")

# Show sample data
print("\nSample Data:")
df_processed[['Ticket Description', 'Ticket Type', 'Ticket Priority']].head()

In [ ]:
# Analyze class distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Ticket Type distribution
df_processed['Ticket Type'].value_counts().plot(kind='bar', ax=ax1)
ax1.set_title('Ticket Type Distribution')
ax1.set_xlabel('Ticket Type')
ax1.set_ylabel('Count')
ax1.tick_params(axis='x', rotation=45)

# Ticket Priority distribution
df_processed['Ticket Priority'].value_counts().plot(kind='bar', ax=ax2)
ax2.set_title('Ticket Priority Distribution')
ax2.set_xlabel('Priority')
ax2.set_ylabel('Count')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Print statistics
print("\nClass Distribution Statistics:")
print(f"Ticket Types: {df_processed['Ticket Type'].nunique()}")
print(f"Priority Levels: {df_processed['Ticket Priority'].nunique()}")
print(f"\nTicket Type Counts:")
print(df_processed['Ticket Type'].value_counts())
print(f"\nPriority Counts:")
print(df_processed['Ticket Priority'].value_counts())

## 2. Text Preprocessing Analysis

In [ ]:
# Analyze text preprocessing
print("Text Preprocessing Analysis")
print("=" * 40)

# Get text statistics
text_stats = preprocessor.get_text_statistics(df_processed)
for key, value in text_stats.items():
    print(f"{key}: {value}")

# Show preprocessing examples
print("\nPreprocessing Examples:")
print("-" * 40)

for i in range(min(3, len(df_processed))):
    original = df_processed.iloc[i]['Ticket Description']
    processed = df_processed.iloc[i]['processed_text']
    
    print(f"\nExample {i+1}:")
    print(f"Original: {original[:100]}...")
    print(f"Processed: {processed}")
    print(f"Length reduction: {len(original)} -> {len(processed)} characters")

## 3. Feature Extraction & Vectorization

In [ ]:
# Prepare data splits
print("Preparing data splits...")
X_train, X_test, y_train, y_test = prepare_data_splits(df_processed)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Training labels: {len(np.unique(y_train))}")
print(f"Test labels: {len(np.unique(y_test))}")

In [ ]:
# Vectorize data using TF-IDF
print("Vectorizing text data...")
X_train_vec, X_test_vec, vectorizer = vectorize_data(
    X_train, X_test, 
    vectorizer_type='tfidf',
    save_path='../outputs/vectorizer.pkl'
)

# Analyze features
feature_analysis = analyze_features(vectorizer, df_processed)
print("\nFeature Analysis:")
for key, value in feature_analysis.items():
    if key != 'top_tfidf_features':
        print(f"{key}: {value}")

# Show top TF-IDF features
if 'top_tfidf_features' in feature_analysis:
    print("\nTop TF-IDF Features:")
    for feature, score in feature_analysis['top_tfidf_features'][:10]:
        print(f"{feature}: {score:.4f}")

## 4. Model Training & Evaluation

In [ ]:
# Initialize and train models
print("Training models...")
trainer = ModelTrainer()

# Train all models
training_results = trainer.train_all_models(X_train_vec, y_train)
print("\nTraining Results:")
for model, result in training_results.items():
    print(f"{model}: {result['status']}")

In [ ]:
# Evaluate all models
print("\nEvaluating models...")
evaluation_results = trainer.evaluate_all_models(X_test_vec, y_test)

# Create comparison table
comparison_df = trainer.create_comparison_table()
print("\nModel Performance Comparison:")
print(comparison_df.to_string(index=False))

# Get best model
best_model_name, best_results = trainer.get_best_model()
print(f"\nBest Model: {best_model_name}")
print(f"Best F1 Score: {best_results['f1_score']:.4f}")
print(f"Best Accuracy: {best_results['accuracy']:.4f}")

In [ ]:
# Visualize model performance
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Model comparison bar chart
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
for i, metric in enumerate(metrics):
    ax = axes[i//2, i%2]
    sns.barplot(data=comparison_df, x='Model', y=metric, ax=ax)
    ax.set_title(f'{metric} by Model')
    ax.set_ylim(0, 1)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Confusion Matrix for Best Model
from sklearn.metrics import ConfusionMatrixDisplay

class_names = list(df_processed['Ticket Type'].unique())
conf_matrix = best_results['confusion_matrix']

plt.figure(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=np.array(conf_matrix),
                              display_labels=class_names)
disp.plot(cmap='Blues', values_format='d')
plt.title(f'Confusion Matrix - {best_model_name}')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Print per-class metrics
print("\nPer-Class Performance:")
for class_name, metrics in best_results['classification_report'].items():
    if isinstance(metrics, dict) and 'precision' in metrics:
        print(f"\n{class_name}:")
        print(f"  Precision: {metrics['precision']:.4f}")
        print(f"  Recall: {metrics['recall']:.4f}")
        print(f"  F1 Score: {metrics['f1-score']:.4f}")
        print(f"  Support: {metrics['support']}")

## 5. Priority Prediction Analysis

In [ ]:
# Analyze priority prediction rules
from predict import TicketPredictor

predictor = TicketPredictor()

# Test priority prediction on sample texts
sample_tickets = [
    "URGENT: System is down and customers cannot access their accounts!",
    "I'm having some issues with the login process, can you help?",
    "Could you provide information about your pricing plans?",
    "CRITICAL: Security breach detected, immediate action required!",
    "The application is running slow today, any suggestions?"
]

print("Priority Prediction Analysis:")
print("=" * 50)

for i, ticket in enumerate(sample_tickets, 1):
    priority = predictor.predict_priority(ticket)
    analysis = predictor.analyze_ticket_text(ticket)
    
    print(f"\nTicket {i}:")
    print(f"Text: {ticket}")
    print(f"Predicted Priority: {priority}")
    print(f"Word Count: {analysis['word_count']}")
    print(f"Sentiment: {analysis['sentiment']}")

## 6. Business Impact Analysis

In [ ]:
# Calculate business impact metrics
print("BUSINESS IMPACT ANALYSIS")
print("=" * 40)

# Manual effort reduction
total_tickets = len(df_processed)
correctly_classified = int(best_results['accuracy'] * total_tickets)
manual_effort_reduction = (correctly_classified / total_tickets) * 100

print(f"1. Classification Performance:")
print(f"   - Total tickets processed: {total_tickets:,}")
print(f"   - Correctly classified: {correctly_classified:,}")
print(f"   - Manual effort reduction: {manual_effort_reduction:.1f}%")

# Priority distribution impact
priority_dist = df_processed['Ticket Priority'].value_counts()
high_priority_pct = (priority_dist.get('Critical', 0) + priority_dist.get('High', 0)) / total_tickets * 100

print(f"\n2. Priority Distribution:")
print(f"   - High/Critical priority: {high_priority_pct:.1f}%")
print(f"   - Automatic routing capability: Available")
print(f"   - Response time improvement: Estimated 30-50% for urgent tickets")

# Processing time savings
avg_manual_time = 5  # minutes per ticket for manual categorization
avg_automated_time = 0.5  # minutes per ticket for automated processing
time_saved_per_ticket = avg_manual_time - avg_automated_time
total_time_saved = time_saved_per_ticket * correctly_classified

print(f"\n3. Time Savings:")
print(f"   - Time saved per ticket: {time_saved_per_ticket:.1f} minutes")
print(f"   - Total time saved: {total_time_saved:.1f} minutes ({total_time_saved/60:.1f} hours)")
print(f"   - Daily capacity increase: ~{total_time_saved/60/8:.1f} staff hours")

## 7. Interactive Prediction Demo

In [ ]:
# Load the best model for interactive predictions
print("Loading best model for predictions...")

# Save the best model
trainer.save_model(best_model_name, '../outputs/best_model.pkl')

# Initialize predictor with trained model
predictor = TicketPredictor(
    model_path='../outputs/best_model.pkl',
    vectorizer_path='../outputs/vectorizer.pkl'
)

# Test with sample tickets
test_tickets = [
    "I can't log into my account, the system keeps showing an error message",",
    "Could you please explain how to set up the new feature?",
    "URGENT: The payment gateway is down and we're losing sales!",
    "Thank you for the great service, everything works perfectly!",
    "I have a suggestion for improving the user interface"
]

print("\nInteractive Prediction Demo:")
print("=" * 40)

for i, ticket in enumerate(test_tickets, 1):
    try:
        result = predictor.predict_complete(ticket)
        
        print(f"\nTicket {i}:")
        print(f"Text: {ticket}")
        print(f"Predicted Category: {result['predicted_category']}")
        print(f"Predicted Priority: {result['predicted_priority']}")
        print(f"Confidence: {result['confidence']:.4f}")
    except Exception as e:
        print(f"Error processing ticket {i}: {str(e)}")

## 8. Model Performance Summary

In [ ]:
# Generate comprehensive evaluation report
evaluator = ModelEvaluator('../outputs')

# Complete evaluation
output_files = evaluator.evaluate_complete(evaluation_results, class_names)

print("Evaluation Complete!")
print("Generated Files:")
for file_type, file_path in output_files.items():
    print(f"  {file_type}: {file_path}")

# Save model results
trainer.save_results('../outputs/model_results.json')

print("\nFinal Summary:")
print("=" * 40)
print(f"Best Model: {best_model_name}")
print(f"Accuracy: {best_results['accuracy']:.4f}")
print(f"F1 Score: {best_results['f1_score']:.4f}")
print(f"Total Features: {len(vectorizer.get_feature_names())}")
print(f"Training Samples: {len(X_train_vec)}")
print(f"Test Samples: {len(X_test_vec)}")

## 9. Recommendations & Next Steps

In [ ]:
print("RECOMMENDATIONS & NEXT STEPS")
print("=" * 40)

print("\n1. Immediate Actions:")
print("   - Deploy the {best_model_name} model in production")
print("   - Implement human review for first 2 weeks")
print("   - Set up monitoring dashboard for model performance")

print("\n2. Model Improvements:")
print("   - Collect more data for underrepresented classes")
print("   - Experiment with advanced models (BERT, RoBERTa)")
print("   - Implement ensemble methods for better performance")

print("\n3. Business Integration:")
print("   - Integrate with existing ticketing system")
print("   - Set up automated routing based on predictions")
print("   - Create SLA monitoring for high-priority tickets")

print("\n4. Long-term Strategy:")
print("   - Retrain models quarterly with new data")
print("   - Implement A/B testing for model updates")
print("   - Expand to multilingual support")
print("   - Add sentiment analysis for customer satisfaction")

print("\n5. Success Metrics:")
print("   - Target: 90%+ accuracy within 6 months")
print("   - Reduce manual categorization by 80%")
print("   - Decrease average response time by 40%")
print("   - Improve customer satisfaction scores by 25%")

print("\n" + "=" * 40)
print("SYSTEM READY FOR PRODUCTION DEPLOYMENT! ")
print("=" * 40)